### Gaussian Processes implemented in JAX

In this notebook we will aim to clearly explain the impact of Kernels in Gaussian Processes implemented in JAX

### Introduction to Gaussian Processes

A Gaussian process is a collection of random variables, any Gaussian process finite number of which have a joint Gaussian distribution.
A GP is completely defined by its mean function :  $$m(x) =  \mathbb{E}[f(x)] = 0 $$ 

and by its covariance fonction $$ k(x,x') = \mathbb{E}[(f(x) - m(x))((f(x') - m(x')))] = 0 $$

We will consider a GP as $$ \mathbb f(x) ∼ GP(m(x), k(x,x')) $$

### Class Definition
We have a GPR class that considers the important elements that will intervene in our differents concepts, such as Covariance Matrices(Training and Testing ones), and the mean function.

In [ ]:
import jax.numpy as jnp
import jax
class GPR:
    def __init__(self, Kernel, Alpha, NormalizeObs=False):
        self.Kernel = Kernel 
        self.Alpha = Alpha
        self.NormalizeObs = NormalizeObs
        self.TrainingData = None
        self.Obs = None
        self.L_ = None
        self.Alpha_ = None
        self.ObsMean = None
        self.ObsSTD = None

### Marginal log likelihood 
Basically the marginal likelihood only role is to give us the best hyperparameters for our given kernel and a given prior : 
$$ \log p(y \mid X, \theta) = -\frac{1}{2} y^\top (K + \alpha I)^{-1} y - \frac{1}{2} \log |K + \alpha I| - \frac{n}{2} \log(2\pi) $$

X corresponds to the training data and y to the observations(it's important to note that the observations in the equation are values and not random variables.), $\theta$ corresponds to the hyperparameters that we want to optimize.
We will take the hyperparameters that maximize the value of the marginal log likelihood

In [ ]:
def marginal_log_likelihood(Kernel, X, Obs, Alpha):
        n = X.shape[0]
        K = Kernel(X)                      
        K = K + Alpha * jnp.eye(n)            
        L = jnp.linalg.cholesky(K)            
        #Resolution of L @ L.T @ Alpha_ = y
        Alpha_ = jax.scipy.linalg.cho_solve((L, True), Obs)
        #Log‑likelihood
        log_lik = -0.5 * jnp.sum(Obs * Alpha_) - jnp.sum(jnp.log(jnp.diag(L))) - 0.5 * n * jnp.log(2 * jnp.pi)
        return log_lik
